# AC-HWGATE: Indian Sign Language (ISL) Master Cloud Training Pipeline

**Model:** AC-HWGATE (Adaptive Context-Aware Hierarchical Windowed Graph Attention Transformer Encoder)
**Dataset:** Indian Sign Language 40-Class Temporal Benchmark (ISL500 + Vidit031)
**Hardware:** Google Colab GPU (T4 / V100 / A100) with PyTorch AMP FP16

This notebook provides the complete reproducible training pipeline for AC-HWGATE, featuring:
- Automated dataset acquisition from Hugging Face
- 32-frame MediaPipe landmark sequences with 6-channel motion dynamics (Position + Velocity + Acceleration)
- Signer-independent disjoint splits (Zero video and zero signer leakage)
- Tiny overfit sanity verification
- 7 systematic training experiments (EXP_A through EXP_G)
- ONNX export with bit-level parity verification (< 1e-4)
- Class prototype embedding extraction for multi-signal unknown gesture rejection


In [ ]:
# 1. Environment Setup & GPU Check
!nvidia-smi
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q onnx onnxruntime scikit-learn pandas numpy matplotlib pyyaml h5py tqdm mediapipe


In [ ]:
# 2. Clone Repository or Setup Directory Structure
import os
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

!git clone https://github.com/nav-in27/sign-language-translator-using-HWGATE.git /content/isl-translator || true
%cd /content/isl-translator


In [ ]:
# 3. Automated Dataset Acquisition
!python scripts/download_isl500.py
!python scripts/build_unified_dataset.py


In [ ]:
# 4. Mandatory Tiny Overfit Sanity Test
!python training/test_overfit_tiny.py


In [ ]:
# 5. Execute Master Training Pipeline & Controlled Experiments
!python train_maximum.py --force


In [ ]:
# 6. Evaluate Test Set & Display Confusion Matrix
from IPython.display import Image, display
import json

with open('reports/final_metrics.json', 'r') as f:
    metrics = json.load(f)
print('=== FINAL BENCHMARK METRICS ===')
for k, v in metrics.items():
    print(f'{k:25s}: {v}')

display(Image('reports/confusion_matrix.png'))


In [ ]:
# 7. Package Trained Checkpoint & ONNX Deliverables
!zip -r /content/achwgate_production_artifacts.zip models/ checkpoints/ reports/ configs/
print('[+] Artifacts packaged to /content/achwgate_production_artifacts.zip')
